# Metadata

Each cobra Object, such as a metabolite, reaction, and gene, can have metadata, which includes annotations, object history and creators.
This data can be obtain and manipulated via the `metadata` attribute of cobra objects, which is an instance of the `Metadata` class. Addtionally, the `annotation` attribute of cobra objects provides an simplified interface to the metadata for compatiblity with older cobrapy versions.

## History

The `history` attribute, present in the `metadata` attribute of a cobra object provides access to the history of an object, such as its creators, date of creation and modified date.

In [1]:
from cobra.core import Model
from cobra.core.metadata import History, Creator
from datetime import datetime

model = Model(name="Illustrative E. coli Model")

Object creators can be created using the `Creator` class and dates can be provided as a datetime string or directly as a `datetime` object. They can be added directly via constructor.

In [2]:
history = History(
        creators=[
            Creator(
                name="Matthias Koenig",
                organisation="HU",
                email="test@test.com",
            ),
        ],
        created_date="2020-06-26T02:34:30+05:30",
        modified_dates=[
            "2020-06-26T12:34:11+00:00",
            "2020-06-26T00:34:11+05:30",
        ],
    )

model.annotation.history = history

Or by adding them to an existing history object.

In [3]:
new_creator = Creator(
                name="Andreas Draeger",
                organisation="University of Tübingen",
                email="test2@test2.com",
            )

model.annotation.history.creators.append(new_creator)
modified_hdtime = datetime.now()
model.annotation.history.modified_dates.append(modified_hdtime)

## Standardized annotations

Standardized annotations relate scientific identifiers (https://identifiers.org), such as BiGG ids, CHEBI ids, and DOIs, to cobrapy objects. A `StandardizedAnnotation` object consists of a set of resources that specify the identifiers, a qualifier that defines the annotation's relation to the cobrapy object, and optionally nested annotations.

In [4]:
from cobra.core.metadata import Qualifier, Resource, StandardizedAnnotation

annotations = [
    StandardizedAnnotation(
        qualifier=Qualifier.Biological_hasTaxon,
        resources=["http://identifiers.org/taxonomy/511145"]
    ),
    StandardizedAnnotation(
        qualifier=Qualifier.Modelling_is,
        resources=Resource("http://identifiers.org/bigg.model/e_coli_core"),
        annotations=[
            StandardizedAnnotation(
                qualifier=Qualifier.Biological_isDescribedBy,
                resources=[Resource("https://identifiers.org/pubmed/1111111"),
                           Resource.from_data(("eco", "ECO:0000004"))])
        ]
    ),
    StandardizedAnnotation(
        qualifier=Qualifier.Biological_hasPart,
        resources=["https://identifiers.org/kegg.module/M00009"],
    ),
    StandardizedAnnotation(
        qualifier=Qualifier.Biological_hasPart,
        resources=["https://identifiers.org/kegg.module/M00003"],
    )
]
model.add_annotations(annotations)

Standardized annotations can be accessed like a list, or specific annotations can be found using a query.

In [5]:
model.metadata.standardized[0]

Qualifier,Qualifier.Biological_hasTaxon
URI,http://identifiers.org/taxonomy/511145
Namespace,taxonomy
Identifier,511145
Memory address,0x7f0cd953c110


In [6]:
model.metadata.standardized.query('Modelling_', 'qualifier')[0]

Search function: 'Modelling_'


cobra.core.metadata.standardized.StandardizedAnnotation({'qualifier': 'bqm_is', 'resources': ['http://identifiers.org/bigg.model/e_coli_core'], 'annotations': [{'qualifier': 'bqb_isDescribedBy', 'resources': ['https://identifiers.org/pubmed/1111111', 'https://identifiers.org/eco/ECO:0000004']}]})

Additionally, resources, URIs and identifier can be accessed directly from the `metadata.standardized` attribute.

In [7]:
# Methods that start with all_ include all nested annotations,
# so the following only prints nested URIs.
for resource in model.metadata.standardized.all_resources:
    if resource not in model.metadata.standardized.resources:
        print(resource.uri)

https://identifiers.org/pubmed/1111111
https://identifiers.org/eco/ECO:0000004


More complex selection logic can be implemented using the `metadata.standardized.resources_for` method. This method enables filtering based on qualifiers and namespaces. All KEGG Modules that are defined as a part of the biological entity that is represented by the annotation can be obtained as follows:

In [8]:
model.metadata.standardized.resources_for(qualifier=Qualifier.Biological_hasPart, namespace="kegg.module")

[Resource(https://identifiers.org/kegg.module/M00009),
 Resource(https://identifiers.org/kegg.module/M00003)]

A flattened representation of the standardized annotation tree can be generated using `metadata.standardized.to_records`. This can be useful when converting annotations to e.g. a pandas DataFrame.

In [9]:
import pandas as pd

pd.DataFrame.from_records(model.metadata.standardized.to_records())

,qualifier,uri,namespace,identifier,annotation_group,parent_group
0,bqb_hasTaxon,http://identifiers.org/taxonomy/511145,taxonomy,511145,1,0
1,bqm_is,http://identifiers.org/bigg.model/e_coli_core,bigg.model,e_coli_core,2,0
2,bqb_isDescribedBy,https://identifiers.org/pubmed/1111111,pubmed,1111111,3,2
3,bqb_isDescribedBy,https://identifiers.org/eco/ECO:0000004,eco,ECO:0000004,3,2
4,bqb_hasPart,https://identifiers.org/kegg.module/M00009,kegg.module,M00009,4,0
5,bqb_hasPart,https://identifiers.org/kegg.module/M00003,kegg.module,M00003,5,0


## Custom annotations

Custom annotations define key-value/URI pairs that can be used for storing any type of key-value pair data which is not suitable to store anywhere else in the model, where an optional URI provides an explanation of the key-value pair. Standardized annotations should be preferred whenever possible, since they provide more interoperability with other tools. Custom annotations can be accessed through an object's `metadata.custom` attribute, which largely behaves like a python dict.

In [10]:
from cobra.core.metadata import CustomAnnotation

entry1 = CustomAnnotation(
    key="cobra_flag",
    value="starred",
    uri="https://cobrapy.readthedocs.io",
)
entry2 = CustomAnnotation.from_data({
    "key": "answer",
    "value": "42",
    "uri": "https://en.wikipedia.org/wiki/The_Hitchhiker%27s_Guide_to_the_Galaxy",
})

model.add_annotations([entry1, entry2])

model.metadata.custom["answer"]

CustomAnnotation('answer': '42' ('https://en.wikipedia.org/wiki/The_Hitchhiker%27s_Guide_to_the_Galaxy'))

Keys should be unique, but entries can be overwritten:

In [11]:
try:
    model.metadata.custom.add({"key": "cobra_flag", "value": "important!"})
except IndexError:
    print("This is not allowed.")
print(f"Nothing changed: {model.metadata.custom['cobra_flag'].value}")

model.metadata.custom["cobra_flag"] = "important!"
print(f"A new flag: {model.metadata.custom['cobra_flag'].value}")

This is not allowed.
Nothing changed: starred
A new flag: important!


`CustomAnnotation` objects are cobrapy objects, so they can have their own metadata. One could add further, standardized, evidence of the information in the key-value pair.

In [12]:
entry2.add_annotations([
    StandardizedAnnotation(
        qualifier=Qualifier.Modelling_isDescribedBy,
        resources="https://identifiers.org/eco/ECO:0000362")
])
model.metadata.custom["answer"].metadata.standardized[0]

Qualifier,Qualifier.Modelling_isDescribedBy
URI,https://identifiers.org/eco/ECO:0000362
Namespace,eco
Identifier,ECO:0000362
Memory address,0x7f0cd955f490


## Legacy annotation interface
Each cobrapy object provides a legacy annotation interface, accessible throught the `annotation` attribute. This interface provides dict-like access to the resources of standardized annotations. However, since standardized annotations also have qualifiers and can be nested and organized, a dict-like class is unable to provide comprehensive interface. When changing or adding resources using this interface, their qualifiers and order are set using defaults and may not be optimal. This interface is in place to retain compatibility with older cobrapy versions and ease migration to the new metadata interface. It is therefore advised to use `metadata.standardized` when writing new code.

In [13]:
print(f"Taxon: {model.annotation['taxonomy']}")
print(f"Modules: {model.annotation['kegg.module']}")

Taxon: ['511145']
Modules: ['M00003', 'M00009']


Some of the issues with this interface become clear when we try to add an NCBI RefSeq genome to the metadata of the model.

In [14]:
model.annotation["refseq.gcf"] = "GCF_000005845.2"
print(model.annotation["refseq.gcf"])
model.metadata.standardized[0]

['GCF_000005845.2']


Qualifier,Qualifier.Biological_is
URI,https://identifiers.org/refseq.gcf/GCF_000005845.2
Namespace,refseq.gcf
Identifier,GCF_000005845.2
Memory address,0x7f0cd9569110


The genome sequence is added with the qualifier `Qualifier.Biological_is`, whilst `Qualifier.Biological_hasPart` would be more appropriate, since the genome is part of the bacterial cell the model aims to represent. In this case, the annotation can easily be corrected through the `metadata.standardized` interface, but in other cases it could require more effort, since the resource could be added to an existing annotation with the same qualifier.

In [15]:
model.metadata.standardized[0].qualifier = Qualifier.Biological_hasPart
print(model.annotation["refseq.gcf"])
model.metadata.standardized[0]

['GCF_000005845.2']


Qualifier,Qualifier.Biological_hasPart
URI,https://identifiers.org/refseq.gcf/GCF_000005845.2
Namespace,refseq.gcf
Identifier,GCF_000005845.2
Memory address,0x7f0cd9569110
